# UIDE · Visión por Computador · Laboratorio 03
## Segmentación de instancias: de cajas a máscaras con YOLO-seg

**Duración:** 20 minutos · **Nivel:** Maestría en IA Aplicada · **Entorno:** Google Colab (CPU o T4)  
**Versión:** 1.0 · **Caso transversal:** inspección visual industrial.

### Objetivos
- Ejecutar segmentación de instancias con un modelo preentrenado.
- Interpretar máscara, caja, clase y confianza.
- Calcular área en píxeles de una máscara y compararla con el área de su caja.
- Explorar el efecto del umbral de confianza.
- Distinguir segmentación semántica (p. ej., U-Net) y de instancias (p. ej., Mask R-CNN, YOLO-seg).

**Importante:** las áreas se expresan en píxeles, no en cm². Para medidas físicas hacen falta calibración y geometría de cámara. El modelo preentrenado en COCO no identifica defectos de banano sin datos específicos.

**Cronograma:** preparación 3 min · inferencia 5 min · áreas 5 min · experimento 4 min · discusión 3 min.

### 1. Preparación
Ejecuta la celda. Descarga pesos del modelo y una fotografía de demostración (requiere conexión la primera vez).

In [ ]:
%pip -q install "ultralytics>=8.3,<9" pillow matplotlib pandas requests

from pathlib import Path
from PIL import Image
import io, requests
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
from ultralytics import YOLO

device = 0 if torch.cuda.is_available() else "cpu"
print("Dispositivo:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU")
modelo = YOLO("yolo11n-seg.pt")
# /content existe en Colab; en local usamos una carpeta escribible en el directorio de trabajo
carpeta = Path("/content/uide_lab03") if Path("/content").is_dir() else Path.cwd() / "uide_lab03"
carpeta.mkdir(exist_ok=True, parents=True)
ruta = carpeta / "bus.jpg"
if not ruta.exists():
    r = requests.get("https://ultralytics.com/images/bus.jpg", timeout=40)
    r.raise_for_status()
    Image.open(io.BytesIO(r.content)).convert("RGB").save(ruta)
imagen = np.array(Image.open(ruta).convert("RGB"))
print("Tamaño imagen (alto, ancho):", imagen.shape[:2])

### 2. Ejecutar segmentación
Observa cómo cada objeto detectado puede tener su propia máscara. La máscara representa píxeles del objeto, mientras la caja delimita una región rectangular.

In [ ]:
res = modelo.predict(source=str(ruta), conf=0.25, imgsz=640,
                     device=device, verbose=False, retina_masks=True)[0]
if res.masks is None or len(res.boxes) == 0:
    raise RuntimeError("No se obtuvieron máscaras. Prueba otra imagen o reduce conf a 0.15.")
plt.figure(figsize=(11,7))
plt.imshow(res.plot()[..., ::-1])
plt.title(f"YOLO-seg: {len(res.boxes)} instancias")
plt.axis("off")
plt.show()
print("Máscaras:", tuple(res.masks.data.shape), "| Cajas:", tuple(res.boxes.xyxy.shape))

### 3. Medir superficies: máscara vs. caja
`retina_masks=True` permite obtener máscaras a resolución de la imagen original. Comprobaremos sus dimensiones antes de contar píxeles.

**Índice de ocupación** = área de máscara / área de caja. No es IoU: la caja y la máscara describen regiones diferentes y no constituyen predicción frente a verdad de terreno.

In [ ]:
alto, ancho = imagen.shape[:2]
mascaras = res.masks.data.detach().cpu().numpy()
if mascaras.shape[1:] != (alto, ancho):
    raise ValueError(f"Máscara {mascaras.shape[1:]} != imagen {(alto, ancho)}; revisar retina_masks.")
cajas = res.boxes.xyxy.detach().cpu().numpy()
clases = res.boxes.cls.detach().cpu().numpy().astype(int)
confianzas = res.boxes.conf.detach().cpu().numpy()

filas = []
for i, (mascara, caja, cls, conf) in enumerate(zip(mascaras, cajas, clases, confianzas)):
    x1,y1,x2,y2 = caja
    area_caja = max(0., x2-x1)*max(0., y2-y1)
    area_mascara = int((mascara > 0.5).sum())
    filas.append({
        "instancia": i, "clase": modelo.names[int(cls)],
        "confianza": round(float(conf),3),
        "area_mascara_px2": area_mascara,
        "area_caja_px2": round(float(area_caja),1),
        "ocupacion": round(area_mascara/area_caja,3) if area_caja else np.nan
    })
tabla = pd.DataFrame(filas)
display(tabla)
print("Recuerda: no son áreas físicas ni métricas de calidad de segmentación.")

### 4. Visualizar una máscara aislada
Selecciona otra instancia cambiando `INDICE`. Si el objeto es irregular, ¿qué información se pierde al representarlo solo mediante una caja?

In [ ]:
INDICE = 0  # prueba 1 o 2 si hay varias instancias
if not 0 <= INDICE < len(mascaras):
    raise IndexError(f"Selecciona un índice entre 0 y {len(mascaras)-1}.")
m = mascaras[INDICE] > 0.5
solo_objeto = imagen.copy()
solo_objeto[~m] = 0
fig, ax = plt.subplots(1,3,figsize=(15,5))
ax[0].imshow(imagen); ax[0].set_title("Imagen original")
ax[1].imshow(m,cmap="gray"); ax[1].set_title("Máscara binaria")
ax[2].imshow(solo_objeto); ax[2].set_title("Objeto aislado")
for a in ax: a.axis("off")
plt.tight_layout(); plt.show()
print(f"Instancia {INDICE}: {tabla.loc[INDICE,'clase']} · {tabla.loc[INDICE,'area_mascara_px2']} píxeles²")

### 5. Experimento: modificar confianza
Cambia los umbrales y observa si varía el número de máscaras. Una máscara no es automáticamente correcta por tener alta confianza.

In [ ]:
UMBRALES = [0.15, 0.50, 0.80]
filas = []
for conf in UMBRALES:
    pred = modelo.predict(source=str(ruta), conf=conf, imgsz=640,
                          device=device, verbose=False, retina_masks=True)[0]
    n = len(pred.boxes)
    area_total = int((pred.masks.data.detach().cpu().numpy() > 0.5).sum()) if pred.masks is not None else 0
    filas.append({"confianza_minima":conf, "instancias":n,
                  "suma_areas_mascaras_px2":area_total})
display(pd.DataFrame(filas))
print("La suma puede contar varias veces píxeles si las máscaras se solapan.")

### 6. Discusión final (3 minutos)

| Arquitectura | Tipo de salida habitual | Aplicación típica |
|---|---|---|
| **U-Net** | Mapa de clases por píxel (semántica) | Delimitar tejidos o zonas defectuosas |
| **Mask R-CNN** | Caja + clase + máscara por objeto (instancias) | Objetos individuales y contornos |
| **YOLO-seg** | Caja + clase + máscara por objeto (instancias) | Segmentación eficiente de objetos |

**Preguntas para discutir en parejas**
1. Si queremos medir qué porcentaje de la superficie de un banano presenta manchas, ¿basta con detectar una caja?
2. ¿Por qué el área de una máscara en píxeles no equivale directamente al área en cm²?
3. ¿Qué datos etiquetados necesitaríamos para entrenar un segmentador de defectos?
4. ¿Qué métrica utilizarías para evaluar máscaras contra anotaciones reales: IoU de máscara o Dice?

**Conclusión:** segmentar permite analizar forma y superficie, pero para un despliegue industrial necesitamos máscaras de referencia, validación en imágenes reales y, si corresponde, calibración.

### Extensión opcional
Para medir calidad de segmentación se necesita un dataset con **máscaras de referencia**. No confundas el índice de ocupación máscara/caja con IoU de evaluación.

**Fuentes:** [Ultralytics Segmentación](https://docs.ultralytics.com/tasks/segment/) · [Ultralytics Predicción](https://docs.ultralytics.com/modes/predict/) · [U-Net (Ronneberger et al.)](https://arxiv.org/abs/1505.04597) · [Mask R-CNN (He et al.)](https://arxiv.org/abs/1703.06870)

**Nota docente:** en la siguiente sesión enlazaremos detección de instancias con seguimiento temporal en video.